# WellMind AI - 4. Inference / Prediction Demo

Loads the saved model and preprocessing pieces from notebooks 1 & 2, then
predicts a Lifestyle Category for one new person, plus a wellness score and recommendations.

In [1]:
import pandas as pd
import numpy as np
import joblib

RANDOM_STATE = 42


## 1. Load the Saved Model & Preprocessing Pieces

In [2]:
import sys
from pathlib import Path

_nb = Path.cwd().resolve()
sys.path.insert(0, str(_nb if (_nb / "paths.py").exists() else _nb / "notebook"))
from paths import (
    BEST_MODEL_PKL, TARGET_ENCODER_PKL, ENCODERS_PKL, SCALER_PKL,
    NUM_FEATURE_COLS_PKL, FEATURE_COLS_PKL,
)

best_model = joblib.load(BEST_MODEL_PKL)
target_encoder = joblib.load(TARGET_ENCODER_PKL)
encoders = joblib.load(ENCODERS_PKL)
scaler = joblib.load(SCALER_PKL)
num_feature_cols = joblib.load(NUM_FEATURE_COLS_PKL)
feature_cols = joblib.load(FEATURE_COLS_PKL)

cat_feature_cols = ["Gender", "Occupation", "BMI Category", "Sleep Disorder"]

print("Model loaded. Classes:", list(target_encoder.classes_))


Model loaded. Classes: ['Average', 'Healthy', 'Poor']


## 2. Enter a New Person's Data

In [3]:
new_person = pd.DataFrame([{
    "Age": 29,
    "Gender": "Female",
    "Occupation": "Teacher",
    "Sleep Duration": 6.2,
    "Quality of Sleep": 6,
    "Physical Activity Level": 40,
    "Stress Level": 7,
    "BMI Category": "Overweight",
    "Daily Steps": 4800,
    "Sleep Disorder": "Insomnia",
}])

new_person


,Age,Gender,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Daily Steps,Sleep Disorder
0,29,Female,Teacher,6.2,6,40,7,Overweight,4800,Insomnia


## 3. Encode & Scale, Same as Training

In [4]:
new_encoded = new_person.copy()

for col in cat_feature_cols:
    le = encoders[col]
    val = new_encoded[col].iloc[0]
    new_encoded[col] = le.transform([val if val in le.classes_ else le.classes_[0]])

new_encoded[num_feature_cols] = scaler.transform(new_encoded[num_feature_cols])

new_encoded


,Age,Gender,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Daily Steps,Sleep Disorder
0,-2.20999,0,10,-1.294591,-1.070154,-1.184489,0.962528,2,-1.498353,1


## 4. Predict the Lifestyle Category

In [5]:
pred = best_model.predict(new_encoded[feature_cols])
pred_label = target_encoder.inverse_transform(pred)[0]

confidence = 0.85
if hasattr(best_model, "predict_proba"):
    proba = best_model.predict_proba(new_encoded[feature_cols])[0]
    confidence = float(np.max(proba))

print("Predicted Lifestyle Category:", pred_label)
print("Confidence:", round(confidence, 4))


Predicted Lifestyle Category: Poor
Confidence: 0.9999


## 5. Compute the Wellness Score (for the dashboard)

In [6]:
def sleep_hours_points(hours):
    if hours < 5:
        return 30
    elif hours < 7:
        return 55
    elif hours <= 9:
        return 90
    else:
        return 70

sleep_score = sleep_hours_points(new_person["Sleep Duration"].iloc[0]) * 0.5 + (new_person["Quality of Sleep"].iloc[0] / 10 * 100) * 0.5

pal = min(max(new_person["Physical Activity Level"].iloc[0], 30), 90)
activity_score = (((pal - 30) / 60) * 100) * 0.5 + min(new_person["Daily Steps"].iloc[0] / 10000 * 100, 100) * 0.5

stress_index = new_person["Stress Level"].iloc[0] * 10
fatigue_score = (100 - sleep_score) * 0.4 + stress_index * 0.35 + (100 - activity_score) * 0.25
wellness_score = sleep_score * 0.35 + activity_score * 0.30 + (100 - stress_index) * 0.20 + (100 - fatigue_score) * 0.15

print("Sleep Score:", round(sleep_score, 1))
print("Activity Score:", round(activity_score, 1))
print("Stress Index:", round(stress_index, 1))
print("Fatigue Score:", round(fatigue_score, 1))
print("Wellness Score:", round(wellness_score, 1))


Sleep Score: 57.5
Activity Score: 32.3
Stress Index: 70
Fatigue Score: 58.4
Wellness Score: 42.1


## 6. Build Detailed Recommendations (8-15 feature-based tips)

In [7]:
def parse_activity_label(pal_value):
    if pal_value <= 40:
        return "Low"
    if pal_value >= 75:
        return "High"
    return "Moderate"

def normalize_bmi(raw):
    s = str(raw).lower()
    if "under" in s:
        return "Underweight"
    if "obese" in s:
        return "Obese"
    if "over" in s:
        return "Overweight"
    return "Normal"

def normalize_disorder(raw):
    s = str(raw).lower()
    if "insomnia" in s:
        return "Insomnia"
    if "apnea" in s:
        return "Sleep Apnea"
    return "None"

recommendations = []

def push(category, tone, text):
    icons = {"positive": "\u2705", "warning": "\u26a0\ufe0f", "neutral": "\u2022"}
    recommendations.append(f"{icons[tone]} [{category}] {text}")

sleep = new_person["Sleep Duration"].iloc[0]
sleep_quality_val = new_person["Quality of Sleep"].iloc[0]
stress_level_val = new_person["Stress Level"].iloc[0]
steps_val = new_person["Daily Steps"].iloc[0]
activity_label = parse_activity_label(new_person["Physical Activity Level"].iloc[0])
bmi_label = normalize_bmi(new_person["BMI Category"].iloc[0])
disorder_label = normalize_disorder(new_person["Sleep Disorder"].iloc[0])

# Sleep Duration
if sleep < 5:
    for t in ["You're not sleeping enough - this can hurt your health.",
              "Try to sleep 2-3 hours more each night.",
              "Turn off screens before bed.",
              "Try going to bed before 10:30 PM."]:
        push("Sleep Duration", "warning", t)
elif sleep < 7:
    for t in ["Try to get about 1 more hour of sleep.",
              "Go to bed at the same time every night.",
              "Skip coffee or tea in the evening."]:
        push("Sleep Duration", "warning", t)
elif sleep <= 9:
    for t in ["You're getting a great amount of sleep!",
              "Keep doing what you're doing.",
              "Stick to your current sleep routine."]:
        push("Sleep Duration", "positive", t)
else:
    push("Sleep Duration", "neutral", "You're sleeping more than most people.")
    push("Sleep Duration", "neutral", "If you still feel tired, it may help to see a doctor.")

# Sleep Quality
if sleep_quality_val <= 5:
    push("Sleep Quality", "warning", "Your sleep isn't very restful - try a darker room and less caffeine late in the day.")
elif sleep_quality_val >= 8:
    push("Sleep Quality", "positive", "You're sleeping well - keep your bedtime routine going.")

# Stress Level
if stress_level_val <= 3:
    push("Stress Level", "positive", "You're handling stress really well.")
    push("Stress Level", "positive", "Keep doing what's working for you.")
elif stress_level_val <= 6:
    for t in ["Take short breaks during your day.",
              "Try some deep breathing when you feel tense.",
              "Make time to rest, not just work."]:
        push("Stress Level", "warning", t)
else:
    for t in ["Your stress level is high right now.",
              "Try a relaxing activity, like a walk or music.",
              "See if you can lighten your workload.",
              "Getting enough sleep can help lower stress too."]:
        push("Stress Level", "warning", t)

# Daily Steps
if steps_val < 4000:
    push("Daily Steps", "warning", "Try to walk a little more each day.")
    push("Daily Steps", "warning", "Aim for 6,000-8,000 steps a day.")
elif steps_val < 8000:
    push("Daily Steps", "warning", "You're doing well - keep it up!")
    push("Daily Steps", "warning", "Try to reach 8,000-10,000 steps a day.")
else:
    push("Daily Steps", "positive", "Great job staying active!")
    push("Daily Steps", "positive", "Keep up your current activity level.")

# Physical Activity
if activity_label == "Low":
    push("Physical Activity", "warning", "Try adding 20-30 minutes of exercise to your day.")
    push("Physical Activity", "warning", "A daily walk is a great place to start.")
elif activity_label == "Moderate":
    push("Physical Activity", "positive", "You have a good activity level.")
    push("Physical Activity", "positive", "Keep exercising regularly.")
else:
    push("Physical Activity", "positive", "You're very active - great job!")
    push("Physical Activity", "positive", "Remember to rest and drink enough water.")

# BMI Category
if bmi_label == "Underweight":
    for t in ["Try eating more nutrient-rich foods.",
              "Add more healthy calories to your meals.",
              "Foods like eggs, fish, beans, milk, and nuts can help."]:
        push("BMI Category", "warning", t)
elif bmi_label == "Normal":
    push("BMI Category", "positive", "You're maintaining a healthy weight.")
    push("BMI Category", "positive", "Keep up your balanced diet and exercise.")
elif bmi_label == "Overweight":
    for t in ["Try cutting back on sugary drinks.",
              "Eat more vegetables and lean protein.",
              "A daily walk can help a lot.",
              "Try to eat less processed food."]:
        push("BMI Category", "warning", t)
else:
    for t in ["It's a good idea to check in with a doctor.",
              "Try to eat a balanced diet.",
              "Slowly add more physical activity to your routine.",
              "Cut back on sugar and fast food where you can."]:
        push("BMI Category", "warning", t)

# Sleep Disorder
if disorder_label == "None":
    push("Sleep Disorder", "positive", "No sleep problems reported - that's great.")
    push("Sleep Disorder", "positive", "Keep up your healthy sleep habits.")
elif disorder_label == "Insomnia":
    for t in ["Try to go to bed at the same time every night.",
              "Avoid caffeine later in the day.",
              "Put screens away before bed."]:
        push("Sleep Disorder", "warning", t)
else:
    for t in ["It may help to see a doctor about this.",
              "Maintaining a healthy weight can help.",
              "Sleeping on your side may help, if your doctor recommends it."]:
        push("Sleep Disorder", "warning", t)

# Keep at most 15 tips
recommendations = recommendations[:15]

print(f"Generated {len(recommendations)} recommendations:\n")
for r in recommendations:
    print(r)


Generated 15 recommendations:

⚠️ [Sleep Duration] Try to get about 1 more hour of sleep.
⚠️ [Sleep Duration] Go to bed at the same time every night.
⚠️ [Sleep Duration] Skip coffee or tea in the evening.
⚠️ [Stress Level] Your stress level is high right now.
⚠️ [Stress Level] Try a relaxing activity, like a walk or music.
⚠️ [Stress Level] See if you can lighten your workload.
⚠️ [Stress Level] Getting enough sleep can help lower stress too.
⚠️ [Daily Steps] You're doing well - keep it up!
⚠️ [Daily Steps] Try to reach 8,000-10,000 steps a day.
⚠️ [Physical Activity] Try adding 20-30 minutes of exercise to your day.
⚠️ [Physical Activity] A daily walk is a great place to start.
⚠️ [BMI Category] Try cutting back on sugary drinks.
⚠️ [BMI Category] Eat more vegetables and lean protein.
⚠️ [BMI Category] A daily walk can help a lot.
⚠️ [BMI Category] Try to eat less processed food.


## 7. Final Result Summary

In [8]:
result = {
    "prediction": f"{pred_label} Lifestyle",
    "wellness_score": int(round(wellness_score)),
    "confidence": round(confidence, 4),
    "recommendations": recommendations,
}

result


{'prediction': 'Poor Lifestyle',
 'wellness_score': 42,
 'confidence': 0.9999,
 'recommendations': ['⚠️ [Sleep Duration] Try to get about 1 more hour of sleep.',
  '⚠️ [Sleep Duration] Go to bed at the same time every night.',
  '⚠️ [Sleep Duration] Skip coffee or tea in the evening.',
  '⚠️ [Stress Level] Your stress level is high right now.',
  '⚠️ [Stress Level] Try a relaxing activity, like a walk or music.',
  '⚠️ [Stress Level] See if you can lighten your workload.',
  '⚠️ [Stress Level] Getting enough sleep can help lower stress too.',
  "⚠️ [Daily Steps] You're doing well - keep it up!",
  '⚠️ [Daily Steps] Try to reach 8,000-10,000 steps a day.',
  '⚠️ [Physical Activity] Try adding 20-30 minutes of exercise to your day.',
  '⚠️ [Physical Activity] A daily walk is a great place to start.',
  '⚠️ [BMI Category] Try cutting back on sugary drinks.',
  '⚠️ [BMI Category] Eat more vegetables and lean protein.',
  '⚠️ [BMI Category] A daily walk can help a lot.',
  '⚠️ [BMI Category